# v3 model demonstration: supervisor meeting

A one-year-per-step run of the lithium supply chain from 2024 to 2050 with the first firm behaviour: firms commit pipeline mines when the price covers their full cost plus a hurdle. Mining firms are either one per mine or one per country (AUS, ARG, CHL, CHN, CAN, USA, and ROW for the rest), set in **Settings**.

**How to use:** change the values in **Settings**, then *Run All*. Every chart is rebuilt from a fresh model. The comparison section at the end re-runs the model for several hurdle rates, demand scenarios, initialisation seeds and mining-agent setups.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

working_directory = Path.cwd().resolve()
model_root = next(
    parent / "02_Model"
    for parent in (working_directory, *working_directory.parents)
    if (parent / "02_Model" / "v3" / "src").is_dir()
)
if str(model_root) not in sys.path:
    sys.path.insert(0, str(model_root))

from v3.src import create_mines, params, units
from v3.src.iea_demand import load_iea_demand
from v3.src.model import mine_sales_by_country
from v3.src.params import Scenario
from v3.src.setup_model import build_model

## Settings

Change these and re-run the notebook.

In [ ]:
SETTINGS = {
    "initialisation_seed": 1,           # redraws the synthetic parts of the mine dataset
    "simulation_seed": 1,               # seeds each project's yearly commitment draws
    "mining_agents": "mine",            # "mine": one firm per mine; "country": one firm per agent country
    "demand_scenario": "STEPS",         # IEA scenario: "CPS", "STEPS" or "HDS"
    "end_year": 2050,
    "operating_capacity_multiplier": 1.3,
    "ledger_year": 2035,                # year shown in the lithium flow diagram
    "scenario": Scenario(
        refined_price_per_t=25_000.0,   # refiners' maximum: caps the price
        scarcity_premium=1.0,           # price = 90th-percentile cost x (1 + premium x unmet share)
        investment_hurdle=0.2,          # commit if price >= full cost x (1 + hurdle)
        commitment_probability=0.5,     # chance an economic project commits in a given year
        capex_payback_years=20.0,       # capex recovered over this many years of output
    ),
}

## Model runner

`run_model` builds a fresh model and steps it one year at a time. After each year it records every mine's status and capacity, and the lithium held in each part of the ledger. These records feed the charts below.

In [ ]:
RAW_MINES = create_mines.load_raw_mines()
IEA = load_iea_demand()
PRODUCTION_2024_T_LI = 222_000.0     # USGS world production, used as 2024 demand
MANUFACTURING_EFFICIENCY = 0.95      # matches setup_model.py
_datasets = {}


def mine_dataset(seed):
    if seed not in _datasets:
        _datasets[seed] = create_mines.create_mines_dataset(RAW_MINES, seed=seed)
    return _datasets[seed]


def demand_for(scenario_name, end_year):
    # Model demand in final-product proxy tonnes (t Li x LCE factor), keyed by period from 2024.
    t_li = IEA.loc[IEA["scenario"].eq(scenario_name)].set_index("year")["demand_t_li"]
    t_li.loc[params.BASE_YEAR] = PRODUCTION_2024_T_LI
    years = range(params.BASE_YEAR, end_year + 1)
    return {year - params.BASE_YEAR: float(t_li[year] * units.LCE_PER_LI) for year in years}


def li_capacity_t(mine):
    # Recovered lithium the mine can produce per year.
    return mine.capacity * mine.composition["Li"] * mine.recovery


def full_cost_per_t_lce(mine, scenario):
    # Full mine cost (opex + capex over the payback period) plus refining, per tonne of LCE.
    route = scenario.routes[mine.output_form]
    lce_per_t_product = mine.target_output_grade * route.lithium_recovery_fraction * units.LCE_PER_LI
    return mine.full_unit_cost(scenario.capex_payback_years) / lce_per_t_product + route.processing_cost_per_t_output


def ledger_group(model, holder_id, stock, form):
    # Readable name for a ledger account.
    stage = model.assets_by_id[holder_id].supply_chain_stage
    if stage == "mining":
        if stock == "geological_stock":
            return "Deposits"
        if stock == "residue":
            return "Mining residue"
        return "Spodumene concentrate" if form == "spodumene_concentrate" else "Brine concentrate"
    if stage == "refining":
        if stock == "residue":
            return "Refining residue"
        if stock == "processing_inputs":
            return "Reagents"
        return "Battery-grade lithium" if form == "battery_grade_li" else "Refinery feed"
    if stage == "manufacturing":
        if stock == "residue":
            return "Manufacturing residue"
        return "Cathode product" if form == "cathode_cell" else "Manufacturer input"
    return "In use"


def run_model(settings, record=True):
    scenario = settings["scenario"]
    demand = demand_for(settings["demand_scenario"], settings["end_year"])
    model = build_model(
        mine_dataset(settings["initialisation_seed"]),
        demand=demand,
        refinery_capacity=max(demand.values()) / MANUFACTURING_EFFICIENCY,
        scenario=scenario,
        simulation_seed=settings["simulation_seed"],
        start_year=params.BASE_YEAR,
        mining_agents=settings["mining_agents"],
    )
    for mine in model.mines:
        if mine.status == "operation":
            mine.capacity *= settings["operating_capacity_multiplier"]

    pipeline = pd.DataFrame([
        {"asset_id": m.asset_id, "deposit_type": m.deposit_type, "capacity_kt_li": li_capacity_t(m) / 1e3,
         "full_cost_per_t_lce": full_cost_per_t_lce(m, scenario), "mine": m}
        for m in model.mines if m.status == "pre-construction"
    ])

    mine_rows, stock_rows = [], []
    for _ in range(len(demand)):
        row = model.step()
        if not record:
            continue
        for mine in model.mines:
            producing = mine.status == "operation" and mine.reserves > 0
            status = "Operating" if producing else {"construction": "Under construction",
                                                    "pre-construction": "Pipeline"}.get(mine.status, "Depleted or closed")
            mine_rows.append({"year": row["year"], "status": status, "capacity_kt_li": li_capacity_t(mine) / 1e3,
                              "country": model.owner[mine.asset_id].home_region})
        for (holder_id, stock, form, component), mass in model.ledger.snapshot().items():
            if component == "Li":
                stock_rows.append({"year": row["year"], "group": ledger_group(model, holder_id, stock, form), "kt_li": mass / 1e3})

    results = pd.DataFrame(model.period_rows)
    results["share_met"] = results["delivered_t"] / results["demand_t"]
    results["new_commitments"] = results["committed_mine_ids"].map(len)
    results["mines_opened"] = results["opened_mine_ids"].map(len)
    mines = pd.DataFrame(mine_rows)
    if record:
        producing = mines[mines["status"].eq("Operating")].groupby("year").size()
        results["mines_operating"] = results["year"].map(producing).fillna(0).astype(int)
    pipeline["committed_year"] = pipeline["mine"].map(lambda m: m.committed_year)
    return {"model": model, "results": results, "mines": mines, "stocks": pd.DataFrame(stock_rows), "pipeline": pipeline}


base = run_model(SETTINGS)
results = base["results"]
results[["year", "price_usd_per_t_lce", "share_met", "new_commitments", "mines_opened", "mines_operating"]].round(
    {"price_usd_per_t_lce": 0, "share_met": 2})

## Key results

Demand against deliveries, the prices mines receive for each product, how much each route supplies, and the lithium accumulating in use.

In [4]:
route_labels = {"spodumene_concentrate": "Hard-rock concentrate", "concentrated_brine": "Concentrated brine"}
route_colours = {"spodumene_concentrate": "#2166ac", "concentrated_brine": "#e69f00"}
kt_li = 1000.0 * units.LCE_PER_LI  # model tonnes (t Li x LCE factor) per kt Li

trades = pd.DataFrame(base["model"].transactions)
upstream = trades.loc[trades["material_form"].isin(route_labels)]
year_of = results.set_index("period")["year"]
prices = (upstream.groupby(["period", "material_form"])["price_per_t_input"].first().unstack("material_form")
          .reindex(index=results["period"], columns=list(route_labels)).set_axis(results["year"], axis=0))
volumes = (upstream.groupby(["period", "material_form"])["output_t"].sum().unstack("material_form")
           .reindex(index=results["period"], columns=list(route_labels)).fillna(0.0).set_axis(results["year"], axis=0))

fig = make_subplots(rows=2, cols=2, vertical_spacing=0.18, subplot_titles=(
    "Final demand and deliveries", "Mine-gate prices", "Purchased supply by route", "Accumulated end-use stock"))
for column, label, colour in [("demand_t", "Demand", "#555555"), ("delivered_t", "Delivered", "#228833")]:
    fig.add_trace(go.Scatter(x=results["year"], y=results[column] / kt_li, name=label, mode="lines+markers",
                             line={"color": colour}), row=1, col=1)
for material, label in route_labels.items():
    fig.add_trace(go.Scatter(x=results["year"], y=prices[material], name=label, legendgroup=material,
                             mode="lines+markers", line={"color": route_colours[material]}), row=1, col=2)
    fig.add_trace(go.Bar(x=results["year"], y=volumes[material] / kt_li, name=label, legendgroup=material,
                         showlegend=False, marker_color=route_colours[material]), row=2, col=1)
fig.add_trace(go.Scatter(x=results["year"], y=results["in_use_t"] / kt_li, name="In use", mode="lines+markers",
                         line={"color": "#884488"}), row=2, col=2)
fig.update_yaxes(title_text="kt Li", row=1, col=1)
fig.update_yaxes(title_text="USD per t product", row=1, col=2)
fig.update_yaxes(title_text="kt Li, refined-output equivalent", row=2, col=1)
fig.update_yaxes(title_text="kt Li", row=2, col=2)
fig.update_xaxes(title_text="Year", dtick=5)
fig.update_yaxes(rangemode="tozero")
fig.update_layout(title="Key results", template="plotly_white", height=750, barmode="stack", hovermode="x unified")
fig.show()

## 1. New mines

Projects committed each year (they pass the price test *and* the yearly commitment chance), mines opening each year, and the number of mines producing.

In [5]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=results["year"], y=results["new_commitments"], name="Committed", marker_color="#e69f00"))
fig.add_trace(go.Bar(x=results["year"], y=results["mines_opened"], name="Opened", marker_color="#2166ac"))
fig.add_trace(go.Scatter(x=results["year"], y=results["mines_operating"], name="Mines operating",
                         mode="lines", line={"color": "#333333", "width": 3}), secondary_y=True)
fig.update_yaxes(title_text="Mines per year", secondary_y=False, rangemode="tozero")
fig.update_yaxes(title_text="Mines operating", secondary_y=True, rangemode="tozero")
fig.update_xaxes(title_text="Year", dtick=5)
fig.update_layout(title="Commitments, openings and operating mines", template="plotly_white", barmode="group",
                  hovermode="x unified", height=450)
fig.show()

## 2. Price and the boom-bust

Top: the price per tonne of refined output (LCE) against the share of demand met. As the shortage deepens the scarcity premium lifts the price, which triggers commitments; mines arrive after their lead times, the shortage closes and the price falls back. Bottom: the mine-gate prices that follow from it for each product.

In [6]:
scenario = SETTINGS["scenario"]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    specs=[[{"secondary_y": True}], [{}]],
                    subplot_titles=("Price and share of demand met", "Mine-gate prices"))
fig.add_trace(go.Scatter(x=results["year"], y=results["price_usd_per_t_lce"], name="Price (USD/t LCE)",
                         line={"color": "#b2182b", "width": 3}), row=1, col=1)
fig.add_trace(go.Scatter(x=results["year"], y=results["share_met"] * 100, name="Demand met (%)",
                         line={"color": "#228833", "width": 3, "dash": "dot"}), row=1, col=1, secondary_y=True)
fig.add_hline(y=scenario.refined_price_per_t, line_dash="dash", line_color="#999999",
              annotation_text="Refiners' maximum", row=1, col=1)
products = {"spodumene_concentrate": ("Spodumene concentrate", 0.0279, "#2166ac"),
            "concentrated_brine": ("Concentrated brine", 0.06, "#e69f00")}
for form, (label, grade, colour) in products.items():
    route = scenario.routes[form]
    gate = grade * route.lithium_recovery_fraction * units.LCE_PER_LI * (results["price_usd_per_t_lce"] - route.processing_cost_per_t_output)
    fig.add_trace(go.Scatter(x=results["year"], y=gate, name=label, line={"color": colour}), row=2, col=1)
fig.update_yaxes(title_text="USD per t LCE", row=1, col=1, secondary_y=False, rangemode="tozero")
fig.update_yaxes(title_text="% of demand met", row=1, col=1, secondary_y=True, range=[0, 105])
fig.update_yaxes(title_text="USD per t product", row=2, col=1, rangemode="tozero")
fig.update_xaxes(title_text="Year", dtick=5, row=2, col=1)
fig.update_layout(template="plotly_white", height=650, hovermode="x unified")
fig.show()

## 3. Capacity by status

Recovered-lithium capacity of mines that are producing, under construction, or still in the pipeline. Demand (delivered lithium) is shown for scale; meeting it takes somewhat more mine capacity because refining and manufacturing lose 10-20 %.

In [7]:
capacity = (base["mines"].groupby(["year", "status"])["capacity_kt_li"].sum().unstack(fill_value=0.0))
status_colours = {"Operating": "#2166ac", "Under construction": "#e69f00", "Pipeline": "#bbbbbb"}
fig = go.Figure()
for status, colour in status_colours.items():
    if status in capacity:
        fig.add_trace(go.Scatter(x=capacity.index, y=capacity[status], name=status, stackgroup="capacity",
                                 line={"width": 0.5, "color": colour}, fillcolor=colour))
fig.add_trace(go.Scatter(x=results["year"], y=results["demand_t"] / units.LCE_PER_LI / 1e3, name="Demand",
                         line={"color": "#000000", "width": 3, "dash": "dash"}))
fig.update_layout(title="Mine capacity by status", template="plotly_white", height=450, hovermode="x unified",
                  yaxis_title="kt Li per year", xaxis_title="Year")
fig.update_xaxes(dtick=5)
fig.show()

## 3b. Supply and capacity by country

Left: supply bought from each country's mines, in refined-output equivalent. Right: capacity of mines operating or under construction. Countries with their own agent are set in `params.MINING_AGENT_COUNTRIES`; all other mines belong to ROW. Both charts work with either mining-agent setup, because every mining firm carries its country code. The table gives the supply values every five years.

In [ ]:
COUNTRIES = [*params.MINING_AGENT_COUNTRIES.values(), params.REST_OF_WORLD]
COUNTRY_COLOURS = dict(zip(COUNTRIES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]))

supply = (mine_sales_by_country(base["model"])["output_t"]
          .reindex(index=results["year"], columns=COUNTRIES, fill_value=0.0) / kt_li)
built = (base["mines"].loc[base["mines"]["status"].isin(["Operating", "Under construction"])]
         .groupby(["year", "country"])["capacity_kt_li"].sum().unstack(fill_value=0.0)
         .reindex(columns=COUNTRIES, fill_value=0.0))

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                    subplot_titles=("Supply by country", "Capacity operating or under construction"))
for country in COUNTRIES:
    colour = COUNTRY_COLOURS[country]
    fig.add_trace(go.Bar(x=supply.index, y=supply[country], name=country, legendgroup=country,
                         marker={"color": colour, "line": {"color": "white", "width": 1}}), row=1, col=1)
    fig.add_trace(go.Scatter(x=built.index, y=built[country], name=country, legendgroup=country, showlegend=False,
                             stackgroup="capacity", line={"width": 0.5, "color": colour}, fillcolor=colour), row=1, col=2)
fig.update_yaxes(title_text="kt Li, refined-output equivalent", row=1, col=1, rangemode="tozero")
fig.update_yaxes(title_text="kt Li per year", row=1, col=2, rangemode="tozero")
fig.update_xaxes(title_text="Year", dtick=5)
fig.update_layout(title="Mining by country", template="plotly_white", height=450, barmode="stack",
                  hovermode="x unified")
fig.show()
supply.iloc[::5].round(1)

## 4. The ledger

**4a. Lithium flows in one year.** Every flow is a ledger movement: from deposits into concentrate and residue, through the refinery and manufacturer, into use. Widths are kt Li.

In [8]:
model = base["model"]
period = SETTINGS["ledger_year"] - params.BASE_YEAR
event_period = {event["event_id"]: event["period"] for event in model.ledger.events}
flows = {}
for movement in model.ledger.movements:
    if movement["component"] != "Li" or event_period[movement["event_id"]] != period:
        continue
    source = ledger_group(model, movement["source"].holder_id, movement["source"].stock, movement["source"].material_form)
    target = ledger_group(model, movement["destination"].holder_id, movement["destination"].stock, movement["destination"].material_form)
    flows[(source, target)] = flows.get((source, target), 0.0) + movement["quantity_t"] / 1e3

nodes = list(dict.fromkeys(name for pair in flows for name in pair))
fig = go.Figure(go.Sankey(
    node={"label": nodes, "pad": 18, "thickness": 16},
    link={"source": [nodes.index(s) for s, _ in flows], "target": [nodes.index(t) for _, t in flows],
          "value": list(flows.values())},
    valueformat=".1f", valuesuffix=" kt Li",
))
fig.update_layout(title=f"Lithium flows through the ledger, {SETTINGS['ledger_year']}", height=500)
fig.show()

**4b. Where the extracted lithium sits over time.** The stacked areas are lithium held in every account except the deposits; the dashed line is the lithium removed from deposits. They coincide because the ledger conserves mass in every period.

In [9]:
stocks = base["stocks"].groupby(["year", "group"])["kt_li"].sum().unstack(fill_value=0.0)
opening_deposits = sum(mass for (_, stock, _, component), mass in model.ledger.opening.items()
                       if stock == "geological_stock" and component == "Li") / 1e3
removed = opening_deposits - stocks["Deposits"]
held = stocks.drop(columns=["Deposits", "Reagents"], errors="ignore")
held = held.loc[:, held.max() > 1e-6]
order = ["Mining residue", "Refining residue", "Manufacturing residue", "In use"]
others = [c for c in held if c not in order]
fig = go.Figure()
for group in [c for c in order if c in held] + others:
    fig.add_trace(go.Scatter(x=held.index, y=held[group], name=group, stackgroup="held", line={"width": 0.5}))
fig.add_trace(go.Scatter(x=removed.index, y=removed, name="Removed from deposits",
                         line={"color": "#000000", "width": 2, "dash": "dash"}))
fig.update_layout(title="Cumulative lithium by ledger account", template="plotly_white", height=450,
                  hovermode="x unified", yaxis_title="kt Li", xaxis_title="Year")
fig.update_xaxes(dtick=5)
fig.show()
print(f"Largest gap between removed and held lithium: {(removed - held.sum(axis=1)).abs().max() * 1e3:.2e} t")

Largest gap between removed and held lithium: 4.82e-08 t


## 5. The pipeline cost curve

Every pre-construction project at the start, sorted by full cost (operating cost plus capex over the payback period, plus refining) per tonne of LCE. Width is capacity. Colour shows the year each project committed; grey projects never committed. The dashed line is the highest cost that could pass the hurdle at the peak price.

In [10]:
curve = base["pipeline"].sort_values("full_cost_per_t_lce").reset_index(drop=True)
curve["right"] = curve["capacity_kt_li"].cumsum()
curve["centre"] = curve["right"] - curve["capacity_kt_li"] / 2
built = curve["committed_year"].notna()
fig = go.Figure()
fig.add_trace(go.Bar(
    x=curve.loc[built, "centre"], y=curve.loc[built, "full_cost_per_t_lce"], width=curve.loc[built, "capacity_kt_li"],
    marker={"color": curve.loc[built, "committed_year"], "colorscale": "Viridis", "line": {"width": 0},
            "colorbar": {"title": "Committed"}},
    name="Committed", customdata=curve.loc[built, ["asset_id", "committed_year"]],
    hovertemplate="Asset %{customdata[0]}<br>Committed %{customdata[1]}<br>%{y:,.0f} USD/t LCE<extra></extra>",
))
fig.add_trace(go.Bar(
    x=curve.loc[~built, "centre"], y=curve.loc[~built, "full_cost_per_t_lce"], width=curve.loc[~built, "capacity_kt_li"],
    marker={"color": "#cccccc", "line": {"width": 0}}, name="Never committed",
    customdata=curve.loc[~built, ["asset_id"]],
    hovertemplate="Asset %{customdata[0]}<br>%{y:,.0f} USD/t LCE<extra></extra>",
))
threshold = results["price_usd_per_t_lce"].max() / (1 + scenario.investment_hurdle)
fig.add_hline(y=threshold, line_dash="dash", annotation_text="Peak price ÷ (1 + hurdle)")
fig.update_layout(title="Pipeline projects by full cost", template="plotly_white", height=500, bargap=0,
                  xaxis_title="Cumulative capacity, kt Li per year", yaxis_title="Full cost, USD per t LCE",
                  legend={"orientation": "h", "y": -0.2})
fig.show()

## 6. Comparisons

Each comparison re-runs the model with one setting changed and everything else as in **Settings**. The four panels are the key results: price, share of demand met, new commitments per year and mines operating.

In [11]:
KEY_RESULTS = [("price_usd_per_t_lce", "Price (USD/t LCE)"), ("share_met", "Share of demand met"),
               ("new_commitments", "New commitments per year"), ("mines_operating", "Mines operating")]
COLOURS = ["#2166ac", "#b2182b", "#228833", "#e69f00", "#762a83"]


def key_result_figure(title):
    fig = make_subplots(rows=2, cols=2, subplot_titles=[label for _, label in KEY_RESULTS],
                        vertical_spacing=0.14, horizontal_spacing=0.08)
    fig.update_layout(title=title, template="plotly_white", height=650, hovermode="x unified")
    fig.update_xaxes(dtick=5)
    return fig


def compare(changes, title):
    # changes: {label: settings overrides}; one run per label, overlaid.
    fig = key_result_figure(title)
    for i, (label, overrides) in enumerate(changes.items()):
        res = run_model({**SETTINGS, **overrides})["results"]
        for j, (column, _) in enumerate(KEY_RESULTS):
            fig.add_trace(go.Scatter(x=res["year"], y=res[column], name=label, legendgroup=label, showlegend=j == 0,
                                     line={"color": COLOURS[i % len(COLOURS)]}), row=j // 2 + 1, col=j % 2 + 1)
    fig.show()

### 6a. Hurdle rate

In [12]:
compare({f"Hurdle {h:.0%}": {"scenario": replace(SETTINGS["scenario"], investment_hurdle=h)} for h in (0.0, 0.2, 0.4)},
        "Effect of the investment hurdle")

### 6b. Demand scenario

In [13]:
compare({name: {"demand_scenario": name} for name in ("CPS", "STEPS", "HDS")}, "Effect of the IEA demand scenario")

### 6c. Initialisation seed

The initialisation seed redraws the synthetic parts of the dataset: imputed reserves, deposit types, grades, costs and lead times. The line is the median over the seeds and the band runs from the lowest to the highest run.

In [14]:
SEEDS = range(1, 21)
runs = [run_model({**SETTINGS, "initialisation_seed": seed})["results"].set_index("year") for seed in SEEDS]
fig = key_result_figure(f"Spread across {len(SEEDS)} initialisation seeds")
for j, (column, _) in enumerate(KEY_RESULTS):
    table = pd.concat([run[column] for run in runs], axis=1)
    row, col = j // 2 + 1, j % 2 + 1
    fig.add_trace(go.Scatter(x=table.index, y=table.max(axis=1), line={"width": 0}, showlegend=False,
                             hoverinfo="skip"), row=row, col=col)
    fig.add_trace(go.Scatter(x=table.index, y=table.min(axis=1), fill="tonexty", fillcolor="rgba(33,102,172,0.2)",
                             line={"width": 0}, name="Range across seeds", showlegend=j == 0), row=row, col=col)
    fig.add_trace(go.Scatter(x=table.index, y=table.median(axis=1), line={"color": "#2166ac", "width": 3},
                             name="Median", showlegend=j == 0), row=row, col=col)
fig.show()

### 6d. Mining agents

The same run with one firm per mine and with one firm per country. Each project draws its yearly commitment chance from its own random stream, and offers at equal cost are filled in dataset order, so neither the ownership setup nor the order in which firms act changes the draws. While firms still decide project by project the two lines coincide (the country line covers the per-mine line). Once firms decide at agent level (forecasts and NPV, to-do A2 and A3), the gap between the lines is the effect of ownership.

In [ ]:
compare({"One firm per mine": {"mining_agents": "mine"}, "Country agents": {"mining_agents": "country"}},
        "Effect of the mining-agent setup")